# 15 - Gold Daily Funnel

Computes an ordered daily funnel. Identified users map to `user_id`; anonymous users map to `anonymous:<session_id>`.

This notebook is deterministic and safe to rerun. It synchronizes
the complete business result with Delta `MERGE`, records an audit
run, and refreshes aggregate-to-source lineage.


In [ ]:
%run ./05_bronze_common


In [ ]:
import uuid

from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F


dbutils.widgets.text("catalog", "abc_retail", "Target catalog")
dbutils.widgets.text("job_run_id", "manual", "Notebook or job run ID")

catalog = validate_identifier(dbutils.widgets.get("catalog").strip(), "catalog")
job_run_id = dbutils.widgets.get("job_run_id").strip()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`gold`")


def sync_gold_table(target_table, source_df, keys):
    """Synchronize a deterministic Gold snapshot with an idempotent Delta MERGE."""
    source_df = source_df.dropDuplicates(keys)
    target_df = spark.table(target_table)
    stale_keys = (
        target_df.select(*keys)
        .join(source_df.select(*keys), keys, "left_anti")
        .withColumn("_delete", F.lit(True))
    )
    merge_source = (
        source_df.withColumn("_delete", F.lit(False))
        .unionByName(stale_keys, allowMissingColumns=True)
    )
    condition = " AND ".join(
        f"target.`{column}` = source.`{column}`" for column in keys
    )
    assignments = {
        column: f"source.`{column}`" for column in source_df.columns
    }
    (
        DeltaTable.forName(spark, target_table)
        .alias("target")
        .merge(merge_source.alias("source"), condition)
        .whenMatchedDelete(condition="source._delete")
        .whenMatchedUpdate(
            condition=(
                "NOT source._delete "
                "AND NOT (target.record_hash <=> source.record_hash)"
            ),
            set=assignments,
        )
        .whenNotMatchedInsert(
            condition="NOT source._delete",
            values=assignments,
        )
        .execute()
    )


def replace_gold_lineage(target_table, lineage_df):
    """Keep the current direct source mappings for one aggregate Gold table."""
    lineage_target = DeltaTable.forName(
        spark, f"{catalog}.control.record_lineage"
    )
    lineage_target.delete(F.col("target_table") == F.lit(target_table))
    prepared = (
        lineage_df.select(
            F.lit(target_table).alias("target_table"),
            F.col("target_record_key").cast("string"),
            F.lit(pipeline_run_id).alias("pipeline_run_id"),
            "source_table",
            F.col("source_record_key").cast("string"),
            "source_file",
            "batch_id",
            F.col("source_ingest_ts").cast("timestamp"),
            F.current_timestamp().alias("recorded_ts"),
        )
        .dropDuplicates(
            [
                "target_table",
                "target_record_key",
                "source_table",
                "source_record_key",
                "source_file",
                "batch_id",
            ]
        )
    )
    if prepared.limit(1).count():
        prepared.write.format("delta").mode("append").saveAsTable(
            f"{catalog}.control.record_lineage"
        )


In [ ]:
pipeline_name = "gold_daily_funnel"
pipeline_run_id = str(uuid.uuid4())
start_ts = datetime.now(timezone.utc)
source_table = f"{catalog}.silver.silver_events"
target_table = f"{catalog}.gold.gold_daily_funnel"

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`gold`.`gold_daily_funnel` (
        event_date DATE NOT NULL,
        view_users BIGINT NOT NULL,
        add_to_cart_users BIGINT NOT NULL,
        purchase_users BIGINT NOT NULL,
        record_hash STRING NOT NULL,
        updated_ts TIMESTAMP NOT NULL,
        pipeline_run_id STRING NOT NULL
    )
    USING DELTA
    PARTITIONED BY (event_date)
    TBLPROPERTIES (
        'comment' = 'Daily ordered view to add_to_cart to purchase funnel'
    )
    """
)

upsert_audit(
    catalog, pipeline_run_id, pipeline_name, "full_snapshot", start_ts,
    "RUNNING", [source_table], job_run_id,
)

try:
    events = (
        spark.table(source_table)
        .withColumn(
            "actor_id",
            F.coalesce(
                F.col("user_id"),
                F.concat(F.lit("anonymous:"), F.col("session_id")),
            ),
        )
        .filter(F.col("actor_id").isNotNull())
    )

    first_views = (
        events.filter(F.col("event_type") == "view")
        .groupBy("event_date", "actor_id")
        .agg(F.min("event_time").alias("first_view_ts"))
    )
    first_adds = (
        events.filter(F.col("event_type") == "add_to_cart")
        .join(first_views, ["event_date", "actor_id"], "inner")
        .filter(F.col("event_time") >= F.col("first_view_ts"))
        .groupBy("event_date", "actor_id", "first_view_ts")
        .agg(F.min("event_time").alias("first_add_to_cart_ts"))
    )
    first_purchases = (
        events.filter(F.col("event_type") == "purchase")
        .join(
            first_adds.select(
                "event_date", "actor_id", "first_add_to_cart_ts"
            ),
            ["event_date", "actor_id"],
            "inner",
        )
        .filter(F.col("event_time") >= F.col("first_add_to_cart_ts"))
        .groupBy("event_date", "actor_id")
        .agg(F.min("event_time").alias("first_purchase_ts"))
    )

    actor_funnel = (
        first_views.join(
            first_adds.select(
                "event_date", "actor_id", "first_add_to_cart_ts"
            ),
            ["event_date", "actor_id"],
            "left",
        )
        .join(first_purchases, ["event_date", "actor_id"], "left")
    )
    business = (
        actor_funnel.groupBy("event_date")
        .agg(
            F.countDistinct("actor_id").cast("long").alias("view_users"),
            F.countDistinct(
                F.when(
                    F.col("first_add_to_cart_ts").isNotNull(),
                    F.col("actor_id"),
                )
            ).cast("long").alias("add_to_cart_users"),
            F.countDistinct(
                F.when(
                    F.col("first_purchase_ts").isNotNull(),
                    F.col("actor_id"),
                )
            ).cast("long").alias("purchase_users"),
        )
    )
    source = (
        business.withColumn(
            "record_hash",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.col("view_users"),
                    F.col("add_to_cart_users"),
                    F.col("purchase_users"),
                ),
                256,
            ),
        )
        .withColumn("updated_ts", F.current_timestamp())
        .withColumn("pipeline_run_id", F.lit(pipeline_run_id))
    )
    sync_gold_table(target_table, source, ["event_date"])

    lineage = events.join(
        source.select("event_date"), "event_date", "inner"
    ).select(
        F.col("event_date").cast("string").alias("target_record_key"),
        F.lit(source_table).alias("source_table"),
        F.col("event_id").alias("source_record_key"),
        "source_file",
        "batch_id",
        F.col("ingest_ts").alias("source_ingest_ts"),
    )
    replace_gold_lineage(target_table, lineage)

    rows_read = events.count()
    rows_written = source.count()
    upsert_audit(
        catalog, pipeline_run_id, pipeline_name, "full_snapshot", start_ts,
        "SUCCESS", [source_table], job_run_id,
        end_ts=datetime.now(timezone.utc), rows_read=rows_read,
        rows_written=rows_written, rows_quarantined=0,
    )
except Exception as exc:
    upsert_audit(
        catalog, pipeline_run_id, pipeline_name, "full_snapshot", start_ts,
        "FAILED", [source_table], job_run_id,
        end_ts=datetime.now(timezone.utc), error_message=str(exc)[:4000],
    )
    raise

spark.table(target_table).orderBy("event_date").display()
